# Quickstart: emotion (multi-class ConceptGroup)

Demonstrates Phase 4 — multi-direction `ConceptGroup` with `relationship="mutually_exclusive"`. Three emotions (joy / sadness / anger), each contributing a binary probe + a shared multinomial diagnostic probe across the group. The cross-concept similarity heatmap surfaces whether the three emotions are linearly distinct or collapse to fewer axes.

Dataset: `examples/data/emotion.group.json`, generated by Claude Haiku 4.5 (20 pairs per emotion). Model: pythia-160m.

In [ ]:
import os

os.environ.setdefault("TRANSFORMERLENS_ALLOW_MPS", "1")

from pathlib import Path

from steerkit import ConceptGroup, load, sweep

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
group = ConceptGroup.load(REPO_ROOT / "examples" / "data" / "emotion.group.json")
print(f"loaded ConceptGroup '{group.name}' (relationship={group.relationship})")
for c in group.concepts:
    print(f"  {c.name}: {len(c.contrast_pairs)} pairs — {c.description[:60]}...")

In [ ]:
model = load("EleutherAI/pythia-160m")
print(f"model: {model.model_id} | layers={model.n_layers}")

## `sweep(group, model)`

The headline one-liner. For each concept, fits per-layer probes with a held-out split and selects the best layer. For mutex groups (≥2 concepts), additionally fits a multinomial probe at the layer with highest multi-class accuracy.

In [ ]:
fit = sweep(group, model, cache_dir=REPO_ROOT / "cache")
for name in fit.names():
    p = fit[name]
    print(
        f"{name}: best layer = {p.layer} (depth {p.normalized_depth:.2f}), "
        f"AUC = {p.metrics['auc_test_logistic']:.3f}"
    )
if fit.multinomial is not None:
    print(
        f"\nmultinomial: best layer = {fit.multinomial.layer} "
        f"(depth {fit.multinomial.normalized_depth:.2f}), "
        f"acc_test = {fit.multinomial.metrics.get('accuracy_test', float('nan')):.3f}"
    )

## Layer-selection curves per emotion

Cohen's d shows where in the network each emotion direction is most cleanly separable. Often emotions diverge by depth — e.g., "valence" (joy vs sadness) emerges earlier than fine-grained distinctions.

In [ ]:
import matplotlib.pyplot as plt

for name in fit.names():
    fig = fit.plot_layer_selection(name, by_classifier="cohens_d_logistic", x_axis="normalized_depth", title=name)
    plt.show()

## Cross-concept similarity heatmap

Cosine similarity between the multinomial probe's per-class direction vectors. A diagonal of 1.0 is expected. Off-diagonal patterns reveal:
* near 0 → emotions are linearly distinct axes (good)
* near +1 → two emotions share a direction (collapse / redundancy)
* near −1 → two emotions are opposites of each other (e.g., joy vs sadness)

On a 160M-parameter model with 20 pairs each, expect noisy estimates.

In [ ]:
fit.plot_similarity()

## Steer toward each emotion in turn

Generate the same prompt under each emotion's steering vector. The differences are subtle on tiny models, but the pipeline supports the full intervention API: `addition` (default), `ablate`, `clamp(target=...)`, `amplify(gamma=...)`.

In [ ]:
test_prompt = "Tell me about your day."
print("unsteered:", fit['joy'].steer(model, test_prompt, alpha=0.0, max_new_tokens=25))
for name in fit.names():
    print(f"{name:>8}: {fit[name].steer(model, test_prompt, alpha=2.0, max_new_tokens=25)}")

## Save the GroupFit

A `GroupFit` save is a directory: one `.probe.safetensors` per concept + the multinomial + a `group.json` snapshot. Reloads via `GroupFit.load(directory)`.

In [ ]:
from steerkit import GroupFit

fit.save(REPO_ROOT / "runs" / "emotion_fit")
print("saved to", REPO_ROOT / "runs" / "emotion_fit")
reloaded = GroupFit.load(REPO_ROOT / "runs" / "emotion_fit")
print("reloaded names:", reloaded.names())
print("multinomial classes:", reloaded.multinomial.class_names if reloaded.multinomial else None)